[![img/cloudevel.png](img/cloudevel.png)](https://cloudevel.com)

# Funciones de ventana.

**sql101 · SQL para Ingeniería y Analítica de Datos** · Cloudevel® · Instructor: José Luis Chiquete Valdivieso · Módulo 3: *Análisis*

> 📝 **Versión del alumno.** Las celdas marcadas con ```-- TU CÓDIGO AQUÍ``` y las de *Tu respuesta* son para ti.

En este *notebook* se usarán funciones de ventana para construir rankings, deduplicar registros, comparar periodos y calcular acumulados.

## 🧭 Cómo usar este *notebook*.

* Ejecuta primero la celda de **configuración** y después las demás **en orden** (```Shift + Enter```).
* Las celdas que comienzan con ```%%sql``` son consultas: todo lo que sigue a ```%%sql``` es *SQL* y puede copiarse a otro motor.
* **Ejercicio:** se resuelve en clase. La **práctica adicional** del final del *notebook* (ejercicios y retos) se resuelve fuera de clase, de forma autónoma.
* En las celdas ```-- TU CÓDIGO AQUÍ```, reemplaza la línea ```SELECT NULL AS tu_respuesta;``` por tu consulta. Las de *✍️ Tu respuesta* son para escribir.

## Configuración.

Abre la base de datos del taller en modo de sólo lectura y habilita las celdas ```%%sql```.

In [ ]:
import duckdb

%load_ext sql
%config SqlMagic.displaylimit = 0
%config SqlMagic.displaycon = False

conexion = duckdb.connect('data/thelook.duckdb', read_only=True)
%sql conexion

Las agregaciones con ```GROUP BY``` reducen muchas filas a una por grupo. Las **funciones de ventana** calculan un valor sobre un conjunto de filas relacionadas (la *ventana*) **sin colapsarlas**: cada fila conserva su detalle y recibe además el resultado del cálculo.

![GROUP BY colapsa las filas; la función de ventana conserva cada fila y añade una columna.](img/diagrama_ventana_group_by.png)

Son indispensables en analítica (rankings, acumulados, comparaciones contra el periodo anterior) y en ingeniería de datos (deduplicación, obtener el último registro de cada entidad). Forman parte del estándar desde *SQL:2003*.

https://cloud.google.com/bigquery/docs/reference/standard-sql/window-function-calls

## Sintaxis.

```
función(...) OVER (
  PARTITION BY columnas   -- define los grupos (opcional)
  ORDER BY columnas       -- define el orden dentro de cada grupo (opcional)
  ROWS BETWEEN ...        -- define el marco de filas (opcional)
)
```

Las funciones de ventana se evalúan **después** de ```WHERE```, ```GROUP BY``` y ```HAVING```, por lo que no pueden usarse en esas cláusulas.

## Agregaciones como ventana.

* Cada producto junto con el precio promedio de su categoría, y la diferencia entre ambos.

In [ ]:
%%sql
SELECT name,
  category,
  retail_price,
  ROUND(AVG(retail_price) OVER (PARTITION BY category), 2) AS promedio_categoria,
  ROUND(retail_price - AVG(retail_price) OVER (PARTITION BY category), 2) AS diferencia
FROM products
ORDER BY category, retail_price DESC
FETCH FIRST 20 ROWS ONLY;

* Participación de cada categoría en las ventas totales. ```SUM(...) OVER ()``` sin ```PARTITION BY``` suma sobre todo el resultado. Como las ventanas se evalúan después de ```GROUP BY```, pueden aplicarse sobre agregados.

In [ ]:
%%sql
SELECT p.category,
  ROUND(SUM(oi.sale_price), 2) AS ventas,
  ROUND(SUM(oi.sale_price) / SUM(SUM(oi.sale_price)) OVER () * 100, 2) AS participacion_pct
FROM order_items AS oi
JOIN products AS p ON oi.product_id = p.id
WHERE oi.status = 'Complete'
GROUP BY p.category
ORDER BY ventas DESC;

## Funciones de numeración.

| Función | Empates |
|---|---|
| ```ROW_NUMBER()``` | Numeración única: 1, 2, 3, 4 |
| ```RANK()``` | Mismo número y salto: 1, 2, 2, 4 |
| ```DENSE_RANK()``` | Mismo número sin salto: 1, 2, 2, 3 |

* Los 3 productos más caros de cada categoría. Como una función de ventana no puede usarse en ```WHERE```, se calcula en una *CTE* y se filtra en la consulta exterior.

In [ ]:
%%sql
WITH ranking AS (
  SELECT category,
    name,
    retail_price,
    ROW_NUMBER() OVER (PARTITION BY category ORDER BY retail_price DESC) AS posicion
  FROM products
)
SELECT category, name, retail_price, posicion
FROM ranking
WHERE posicion <= 3
ORDER BY category, posicion;

> **Nota de dialecto:** *DuckDB*, *BigQuery*, *Snowflake* y otros motores ofrecen la cláusula no estándar ```QUALIFY``` para filtrar directamente por el resultado de una ventana, sin *CTE*.

## Patrón de ingeniería: deduplicación y "último registro".

En las cargas de datos es común recibir varias versiones del mismo registro. El patrón estándar para quedarse con una sola fila por entidad es numerar con ```ROW_NUMBER()``` y conservar la fila 1.

* La orden más reciente de cada cliente.

In [ ]:
%%sql
WITH numeradas AS (
  SELECT user_id, order_id, status, created_at,
    ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY created_at DESC, order_id DESC) AS rn
  FROM orders
)
SELECT user_id, order_id, status, created_at
FROM numeradas
WHERE rn = 1
ORDER BY user_id
FETCH FIRST 20 ROWS ONLY;

Observa que ```order_id``` desempata cuando dos órdenes tienen el mismo ```created_at```; sin un desempate, el resultado no sería determinista.

## Funciones de desplazamiento: ```LAG``` y ```LEAD```.

* ```LAG(columna, n)``` obtiene el valor de ```n``` filas antes (por omisión, 1).
* ```LEAD(columna, n)``` obtiene el valor de ```n``` filas después.

* Ventas mensuales y crecimiento contra el mes anterior.

In [ ]:
%%sql
WITH mensual AS (
  SELECT EXTRACT(YEAR FROM created_at) AS anio,
    EXTRACT(MONTH FROM created_at) AS mes,
    SUM(sale_price) AS ventas
  FROM order_items
  WHERE status = 'Complete'
  GROUP BY EXTRACT(YEAR FROM created_at), EXTRACT(MONTH FROM created_at)
),
con_anterior AS (
  SELECT anio, mes, ventas,
    LAG(ventas) OVER (ORDER BY anio, mes) AS ventas_mes_anterior
  FROM mensual
)
SELECT anio,
  mes,
  ROUND(ventas, 2) AS ventas,
  ROUND(ventas_mes_anterior, 2) AS ventas_mes_anterior,
  ROUND((ventas - ventas_mes_anterior) / NULLIF(ventas_mes_anterior, 0) * 100, 1) AS crecimiento_pct
FROM con_anterior
ORDER BY anio, mes;

* Días entre una compra y la anterior del mismo cliente.

In [ ]:
%%sql
WITH compras AS (
  SELECT user_id, order_id, created_at,
    LAG(created_at) OVER (PARTITION BY user_id ORDER BY created_at) AS compra_anterior
  FROM orders
)
SELECT user_id, order_id, created_at, compra_anterior,
  EXTRACT(DAY FROM (created_at - compra_anterior)) AS dias_desde_compra_anterior
FROM compras
WHERE compra_anterior IS NOT NULL
ORDER BY user_id, created_at
FETCH FIRST 20 ROWS ONLY;

## Acumulados y promedios móviles.

Cuando una ventana tiene ```ORDER BY```, las agregaciones se calculan de forma acumulada. El **marco** ```ROWS BETWEEN``` permite indicar exactamente qué filas se consideran.

* Ventas diarias de 2024, acumulado del año y promedio móvil de 7 días.

In [ ]:
%%sql
WITH diario AS (
  SELECT CAST(created_at AS DATE) AS dia,
    SUM(sale_price) AS ventas
  FROM order_items
  WHERE status = 'Complete'
    AND created_at >= TIMESTAMP '2024-01-01 00:00:00'
    AND created_at < TIMESTAMP '2025-01-01 00:00:00'
  GROUP BY CAST(created_at AS DATE)
)
SELECT dia,
  ROUND(ventas, 2) AS ventas,
  ROUND(SUM(ventas) OVER (ORDER BY dia ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) AS acumulado,
  ROUND(AVG(ventas) OVER (ORDER BY dia ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 2) AS promedio_movil_7d
FROM diario
ORDER BY dia
FETCH FIRST 20 ROWS ONLY;

## ✏️ Ejercicios.

Estos **ejercicios** se resuelven en clase. Al final del *notebook* hay una sección de **práctica adicional**, para resolver fuera de clase. Las soluciones están en ```soluciones/```.

### Ejercicio 1 · Ranking por departamento.

Para cada departamento (```department```), obtén las 5 marcas con más productos, usando ```RANK()```.

In [ ]:
%%sql
-- TU CÓDIGO AQUÍ: reemplaza la siguiente línea por tu consulta.
SELECT NULL AS tu_respuesta;

### Ejercicio 2 · La primera orden.

Obtén la **primera** orden de cada cliente (```order_id``` y fecha) usando ```ROW_NUMBER()```.

In [ ]:
%%sql
-- TU CÓDIGO AQUÍ: reemplaza la siguiente línea por tu consulta.
SELECT NULL AS tu_respuesta;

### Ejercicio 3 · Posición dentro de la marca.

Para cada producto, muestra su precio y su posición dentro de su marca (```DENSE_RANK```, del más caro al más barato).

In [ ]:
%%sql
-- TU CÓDIGO AQUÍ: reemplaza la siguiente línea por tu consulta.
SELECT NULL AS tu_respuesta;

## 📝 Práctica adicional.

Esta sección se resuelve **fuera de clase**, de forma autónoma. Solo usa lo ya presentado en esta lección. Las soluciones están en ```soluciones/```.

### Ejercicio 4 · Clientes acumulados.

Calcula el número de clientes nuevos por mes (```users.created_at```) y el total acumulado de clientes.

In [ ]:
%%sql
-- TU CÓDIGO AQUÍ: reemplaza la siguiente línea por tu consulta.
SELECT NULL AS tu_respuesta;

### Reto 1 · Crecimiento por categoría.

Calcula las ventas mensuales por categoría y el crecimiento porcentual de cada categoría respecto a su mes anterior (pista: ```PARTITION BY category```).

In [ ]:
%%sql
-- TU CÓDIGO AQUÍ (opcional): reemplaza la siguiente línea por tu consulta.
SELECT NULL AS tu_respuesta;

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>